# Real Data CPU: Exact-Location Hybrid Lean vs Grid Column V3 Batched

This notebook compares one real GEMS day/month ordering on CPU.

- `Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc`: `keep_ori=True`, so covariance uses `Source_Latitude` / `Source_Longitude`.
- `ColumnV3Batched_Up2_Right3_Down14_Lag2_head0_gridloc`: `keep_ori=False`, so covariance and reverse-L scan use regular `Latitude` / `Longitude` grid coordinates.

Column V3 batched uses the reverse-L/downward-right logic without template reuse. The current candidate uses `above_count=2`, `right_col_count=3`, and `per_lag_conditioning_count=14`; with lags `t, t-1, t-2`, nominal conditioning is `2 + 14 x 3 = 44`, close to Hybrid Lean nominal `41`.

The goal is to test whether the Up2/Right3 conditioning geometry gives sensible real-data estimates for one day before scaling to more days.


In [15]:
import sys, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_vecchia_hybrid import HybridVecchiaFit
from GEMS_TCO.kernel_vecchia_col_batch import ReverseLColumnVecchiaFitBatch

DEVICE = torch.device('cpu')  # Force CPU for local-computer real-data comparison
DTYPE = torch.float64

print('SRC:', SRC)
print('torch:', torch.__version__)
print('device:', DEVICE)

SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
torch: 2.5.1
device: cpu


In [16]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [3]  # 0-based: [2] = July 3. Use list(range(5)) for July 1-5.
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

MM_COND_NUMBER = 100
FIT_SMOOTHS = [0.5]

# Hybrid Lean: original/exact source coordinates for covariance
HYBRID_SPEC = {
    'model': 'Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc',
    'nheads': 0,
    'limit_A': 20,
    'lag1_local_count': 8,
    'lag1_fresh_count': 4,
    'lag2_local_count': 4,
    'lag2_fresh_count': 3,
    'daily_stride': 2,
    'lag1_lon_offset': 0.063,
    # None => HybridVecchiaFit builds shift lookup from the exact t0 coords in keep_ori=True data.
    'spatial_coords_for_shift': None,
}

# Column V3 batched: Up2/Right3 candidate from real-pattern simulation sweep.
# total nominal conditioning = above_count + per_lag_conditioning_count * (lag_count + 1) = 44.
COLUMN_SPEC = {
    'model': 'ColumnV3Batched_Up2_Right3_Down14_Lag2_head0_gridloc',
    'head_right_cols': 0,
    'above_count': 2,
    'right_col_count': 3,
    'per_lag_conditioning_count': 14,
    'lag_count': 2,
    'include_lag_self': False,
    'target_chunk_size': 512 if DEVICE.type == 'cpu' else 4096,
}
COLUMN_TEMPLATE_WARN_GT = 500

LBFGS_LR = 1.0
LBFGS_STEPS = 5
LBFGS_EVAL = 15
LBFGS_HIST = 10

INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']
ROUND_DECIMALS = 4

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'real_exactloc_hybrid_vs_grid_col_ver3_up2r3_cpu_050726_results.csv'

print('days:', DAY_IDX_LIST)
print('output:', OUT_CSV)
print('column_v3:', COLUMN_SPEC)


days: [3]
output: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log/real_exactloc_hybrid_vs_grid_col_ver3_up2r3_cpu_050726_results.csv
column_v3: {'model': 'ColumnV3Batched_Up2_Right3_Down14_Lag2_head0_gridloc', 'head_right_cols': 0, 'above_count': 2, 'right_col_count': 3, 'per_lag_conditioning_count': 14, 'lag_count': 2, 'include_lag_self': False, 'target_chunk_size': 512}


In [17]:
def phys_to_log(d):
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
            d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    p = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq':    np.exp(p[0]) / phi2,
        'range_lat':  rlon / phi3 ** 0.5,
        'range_lon':  rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat':  p[4],
        'advec_lon':  p[5],
        'nugget':     np.exp(p[6]),
    }


def make_params():
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(INIT_DICT)]


def count_valid(day_map):
    return sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in day_map.values())


def empty_device_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def hybrid_tail_count(model):
    total = 0
    for x in (getattr(model, 'X_A', None), getattr(model, 'X_AB', None), getattr(model, 'X_ABC', None)):
        if x is not None:
            total += int(x.shape[0])
    return total


def result_row_common(date_str, day_idx, smooth, model_name, kernel, keep_ori, loss, fit_iter, pre_s, fit_s, n_valid, est):
    row = {
        'date_str': date_str,
        'day_idx': day_idx,
        'smooth': smooth,
        'model': model_name,
        'kernel': kernel,
        'keep_ori': keep_ori,
        'loss': float(loss),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter) + 1,
        'precompute_s': round(pre_s, 3),
        'fit_s': round(fit_s, 3),
        'total_s': round(pre_s + fit_s, 3),
        'n_valid': int(n_valid),
    }
    row.update({f'est_{k}': est[k] for k in P_LABELS})
    return row


def column_template_diagnostics(model):
    batched = getattr(model, 'Batched_Groups', None)
    if batched:
        group_sizes = np.asarray([int(g['target_idx'].shape[0]) for g in batched], dtype=np.int64)
        m_sizes = np.asarray([int(g['max_m']) for g in batched], dtype=np.int64)
        return {
            'n_templates': np.nan,
            'n_batches': int(len(batched)),
            'largest_template_n': int(group_sizes.max()),
            'median_template_n': float(np.median(group_sizes)),
            'mean_template_n': float(group_sizes.mean()),
            'mean_m_by_template': float(m_sizes.mean()),
            'median_m_by_template': float(np.median(m_sizes)),
            'max_m_by_template': int(m_sizes.max()),
        }

    groups = getattr(model, 'Grouped_Batches', [])
    if not groups:
        return {
            'n_templates': 0,
            'n_batches': 0,
            'largest_template_n': 0,
            'median_template_n': 0.0,
            'mean_template_n': 0.0,
            'mean_m_by_template': 0.0,
            'median_m_by_template': 0.0,
            'max_m_by_template': 0,
        }
    group_sizes = np.asarray([int(g['target_idx'].shape[0]) for g in groups], dtype=np.int64)
    m_sizes = np.asarray([int(g['offsets'].shape[0]) for g in groups], dtype=np.int64)
    return {
        'n_templates': int(len(groups)),
        'n_batches': np.nan,
        'largest_template_n': int(group_sizes.max()),
        'median_template_n': float(np.median(group_sizes)),
        'mean_template_n': float(group_sizes.mean()),
        'mean_m_by_template': float(m_sizes.mean()),
        'median_m_by_template': float(np.median(m_sizes)),
        'max_m_by_template': int(m_sizes.max()),
    }

print('Initial log params:', [round(x, 4) for x in phys_to_log(INIT_DICT)])

def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out

def save_results(rows):
    df = pd.DataFrame(rows)
    round_df(df).to_csv(OUT_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    return df


Initial log params: [3.9558, 1.3863, 0.4463, -3.5835, 0.0218, -0.1689, -1.3984]


In [18]:
# Load full month once.
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

sorted_month_keys = sorted(df_map.keys())
day_key_map = {
    day_idx: sorted_month_keys[day_idx * 8:(day_idx + 1) * 8]
    for day_idx in DAY_IDX_LIST
}
selected_keys = [k for day_idx in DAY_IDX_LIST for k in day_key_map[day_idx]]
df_map_selected = {k: df_map[k] for k in selected_keys}

# The grid is fixed within the month, so one selected time slot is enough for coordinates.
first_key = selected_keys[0]
first_df = df_map_selected[first_key].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)
source_coords_t0_ordered = first_df[['Source_Latitude', 'Source_Longitude']].values.astype(np.float64)

# Drop full-month dataframe map after extracting the selected day(s), reducing notebook memory pressure.
del df_map
gc.collect()

print(f'Monthly mean TCO: {monthly_mean:.4f}')
print(f'Month time slots loaded then trimmed: {len(sorted_month_keys)} -> {len(selected_keys)}')
print(f'Grid cells: {len(grid_coords_ordered):,}')
print('grid lat:', grid_coords_ordered[:, 0].min(), grid_coords_ordered[:, 0].max())
print('grid lon:', grid_coords_ordered[:, 1].min(), grid_coords_ordered[:, 1].max())
print('source selected t0 lat:', np.nanmin(source_coords_t0_ordered[:, 0]), np.nanmax(source_coords_t0_ordered[:, 0]))
print('source selected t0 lon:', np.nanmin(source_coords_t0_ordered[:, 1]), np.nanmax(source_coords_t0_ordered[:, 1]))

--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---
Monthly mean TCO: 257.9726
Month time slots loaded then trimmed: 248 -> 8
Grid cells: 18,126
grid lat: -2.9720000000000044 2.0
grid lon: 121.04600000000188 131.0
source selected t0 lat: -2.9828818 1.9996783
source selected t0 lon: 121.014915 130.99995


In [19]:
def fit_hybrid_exact_day(day_idx, smooth):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    day_keys = day_key_map[day_idx]
    day_df_map = {k: df_map_selected[k] for k in day_keys}
    hour_indices = [0, len(day_keys)]

    # keep_ori=True: covariance coordinates are Source_Latitude/Source_Longitude.
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=True,
    )
    day_map = {k: v.to(DEVICE) for k, v in day_map.items()}
    n_valid = count_valid(day_map)

    print('\n' + '=' * 80)
    print(f'HYBRID EXACT | DAY {date_str} | smooth={smooth} | {day_keys[0]} ... {day_keys[-1]} | valid={n_valid:,}')

    params = make_params()
    model = HybridVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        nns_map=nns_map_full,
        mm_cond_number=MM_COND_NUMBER,
        nheads=HYBRID_SPEC['nheads'],
        limit_A=HYBRID_SPEC['limit_A'],
        limit_B_local=HYBRID_SPEC['lag1_local_count'],
        limit_C_local=HYBRID_SPEC['lag2_local_count'],
        daily_stride=HYBRID_SPEC['daily_stride'],
        spatial_coords=HYBRID_SPEC['spatial_coords_for_shift'],
        lag1_lon_offset=HYBRID_SPEC['lag1_lon_offset'],
        lag1_fresh_count=HYBRID_SPEC['lag1_fresh_count'],
        lag2_fresh_count=HYBRID_SPEC['lag2_fresh_count'],
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre

    optimizer = model.set_optimizer(
        params, lr=LBFGS_LR,
        max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL,
        history_size=LBFGS_HIST,
    )
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit

    est = backmap_params(out)
    row = result_row_common(
        date_str, day_idx, smooth, HYBRID_SPEC['model'], 'hybrid_lean_exactloc',
        True, float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est,
    )
    row.update({
        'total_conditioning_nominal': 20 + 1 + 8 + 4 + 1 + 4 + 3,
        'n_heads': int(model.Heads_data.shape[0]),
        'n_tails': hybrid_tail_count(model),
        'n_templates': np.nan,
        'coords_used_for_covariance': 'Source_Latitude/Source_Longitude',
        'coords_used_for_shift_lookup': 'exact_t0_source_coords',
    })
    print('RESULT:', row)

    del model, day_map, params, optimizer
    empty_device_cache()
    return row


def fit_column_grid_day(day_idx, smooth):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    day_keys = day_key_map[day_idx]
    day_df_map = {k: df_map_selected[k] for k in day_keys}
    hour_indices = [0, len(day_keys)]

    # keep_ori=False: covariance and reverse-L scan coordinates are regular Latitude/Longitude grid coordinates.
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=False,
    )
    day_map = {k: v.to(DEVICE) for k, v in day_map.items()}
    n_valid = count_valid(day_map)

    print('\n' + '=' * 80)
    print(f"COLUMN GRID V3 BATCHED | {COLUMN_SPEC['model']} | DAY {date_str} | smooth={smooth} | {day_keys[0]} ... {day_keys[-1]} | valid={n_valid:,}")

    params = make_params()
    model = ReverseLColumnVecchiaFitBatch(
        smooth=smooth,
        input_map=day_map,
        mm_cond_number=MM_COND_NUMBER,
        grid_coords=grid_coords_ordered,
        head_right_cols=COLUMN_SPEC['head_right_cols'],
        above_count=COLUMN_SPEC['above_count'],
        right_col_count=COLUMN_SPEC['right_col_count'],
        per_lag_conditioning_count=COLUMN_SPEC['per_lag_conditioning_count'],
        lag_count=COLUMN_SPEC['lag_count'],
        include_lag_self=COLUMN_SPEC['include_lag_self'],
        target_chunk_size=COLUMN_SPEC['target_chunk_size'],
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = column_template_diagnostics(model)
    try:
        diag['n_spatial_templates'] = int(model.expected_spatial_template_count_upper_bound())
    except Exception:
        diag['n_spatial_templates'] = np.nan
    if pd.notna(diag.get('n_templates', np.nan)) and diag['n_templates'] > COLUMN_TEMPLATE_WARN_GT:
        print(f"WARNING: column templates exploded: {diag['n_templates']} templates. Missing pattern may be breaking reuse.")
    print('COLUMN DIAGNOSTICS:', diag)

    optimizer = model.set_optimizer(
        params, lr=LBFGS_LR,
        max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL,
        history_size=LBFGS_HIST,
    )
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit

    est = backmap_params(out)
    row = result_row_common(
        date_str, day_idx, smooth, COLUMN_SPEC['model'], 'column_reverse_l_v3_gridloc',
        False, float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est,
    )
    row.update({
        'total_conditioning_nominal': COLUMN_SPEC['above_count'] + COLUMN_SPEC['per_lag_conditioning_count'] * (COLUMN_SPEC['lag_count'] + 1),
        'n_heads': int(model.Heads_data.shape[0]),
        'n_tails': int(model.n_tails),
        **diag,
        'coords_used_for_covariance': 'Latitude/Longitude grid',
        'coords_used_for_shift_lookup': 'regular_grid_reverse_l_scan',
        'head_right_cols': COLUMN_SPEC['head_right_cols'],
        'above_count': COLUMN_SPEC['above_count'],
        'right_col_count': COLUMN_SPEC['right_col_count'],
        'per_lag_conditioning_count': COLUMN_SPEC['per_lag_conditioning_count'],
        'lag_count': COLUMN_SPEC['lag_count'],
    })
    print('RESULT:', row)

    del model, day_map, params, optimizer
    empty_device_cache()
    return row

In [20]:
# Run comparison.
# Hybrid is run first and then freed; column is loaded separately with keep_ori=False.
rows = []

for day_idx in DAY_IDX_LIST:
    for smooth in FIT_SMOOTHS:
        rows.append(fit_hybrid_exact_day(day_idx, smooth))
        save_results(rows)
        rows.append(fit_column_grid_day(day_idx, smooth))
        save_results(rows)

results = pd.DataFrame(rows)
print('Saved:', OUT_CSV)
display(results)


HYBRID EXACT | DAY 20240704 | smooth=0.5 | 2024_07_y24m07day04_hm00:53 ... 2024_07_y24m07day04_hm07:48 | valid=139,272
Pre-computing HybridVecchia (smooth=0.5) [A=20, AB=33, ABC=41, B=local8+fresh4, C=local4+fresh3, lag1_offset=0.0630, stored=1]... [Mean Lat: -0.4592] [SetC=True] Done. (Heads=0, Tails A/AB/ABC=18063/17609/103600)
--- Starting Batched L-BFGS Optimization (GPU) ---
--- Step 1/5 / Loss: 1.442278 ---
  Param 0: Value=4.1283, Grad=0.0035910089379529164
  Param 1: Value=1.3202, Grad=-0.0013419836156453938
  Param 2: Value=0.7844, Grad=0.0014054817754637005
  Param 3: Value=-3.9723, Grad=0.0012673581443568111
  Param 4: Value=0.0135, Grad=0.0056614710105920285
  Param 5: Value=-0.1470, Grad=0.010851086201130537
  Param 6: Value=-1.2272, Grad=-0.00230969537571835
  Max Abs Grad: 1.085109e-02
------------------------------
--- Step 2/5 / Loss: 1.424011 ---
  Param 0: Value=4.0278, Grad=0.02273680692128372
  Param 1: Value=1.2042, Grad=-0.003406296013408753
  Param 2: Value=0.3

,date_str,day_idx,smooth,model,kernel,keep_ori,loss,fit_iter_raw,fit_steps_reported,precompute_s,...,mean_template_n,mean_m_by_template,median_m_by_template,max_m_by_template,n_spatial_templates,head_right_cols,above_count,right_col_count,per_lag_conditioning_count,lag_count
0,20240704,3,0.5,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,hybrid_lean_exactloc,True,1.416900,4,5,8.703,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20240704,3,0.5,ColumnV3Batched_Up2_Right3_Down14_Lag2_head0_g...,column_reverse_l_v3_gridloc,False,1.295029,2,3,6.606,...,13.571623,30.804522,33.0,42.0,31.0,0.0,2.0,3.0,14.0,2.0


In [21]:
summary_cols = [
    'date_str', 'smooth', 'model', 'kernel', 'keep_ori',
    'total_conditioning_nominal', 'head_right_cols', 'above_count', 'right_col_count',
    'per_lag_conditioning_count', 'lag_count', 'n_heads', 'n_tails', 'n_batches', 'n_templates', 'n_spatial_templates',
    'largest_template_n', 'median_template_n', 'mean_template_n', 'mean_m_by_template', 'max_m_by_template',
    'loss', 'precompute_s', 'fit_s', 'total_s', 'n_valid',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'est_nugget',
    'coords_used_for_covariance', 'coords_used_for_shift_lookup',
]
existing = [c for c in summary_cols if c in results.columns]
display(round_df(results[existing].sort_values(['date_str', 'smooth', 'kernel'])))

param_rows = []
for _, row in results.iterrows():
    for p in P_LABELS:
        param_rows.append({
            'date_str': row['date_str'],
            'model': row['model'],
            'parameter': p,
            'estimate': row[f'est_{p}'],
        })
param_est = pd.DataFrame(param_rows)
display(param_est)

,date_str,smooth,model,kernel,keep_ori,total_conditioning_nominal,head_right_cols,above_count,right_col_count,per_lag_conditioning_count,...,n_valid,est_sigmasq,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,est_nugget,coords_used_for_covariance,coords_used_for_shift_lookup
1,20240704,0.5,ColumnV3Batched_Up2_Right3_Down14_Lag2_head0_g...,column_reverse_l_v3_gridloc,False,44,0.0,2.0,3.0,14.0,...,139272,11.8445,0.1469,0.1989,1.4958,0.0132,-0.1694,0.9303,Latitude/Longitude grid,regular_grid_reverse_l_scan
0,20240704,0.5,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,hybrid_lean_exactloc,True,41,NaN,NaN,NaN,NaN,...,139272,15.5271,0.2894,0.4268,2.7678,0.0140,-0.1481,1.8687,Source_Latitude/Source_Longitude,exact_t0_source_coords


,date_str,model,parameter,estimate
0,20240704,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,sigmasq,15.527069
1,20240704,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,range_lat,0.289361
2,20240704,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,range_lon,0.426799
3,20240704,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,range_time,2.767794
4,20240704,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,advec_lat,0.014033
5,20240704,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,advec_lon,-0.148136
6,20240704,Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc,nugget,1.868702
7,20240704,ColumnV3Batched_Up2_Right3_Down14_Lag2_head0_g...,sigmasq,11.844527
8,20240704,ColumnV3Batched_Up2_Right3_Down14_Lag2_head0_g...,range_lat,0.146941
9,20240704,ColumnV3Batched_Up2_Right3_Down14_Lag2_head0_g...,range_lon,0.198892


## Why the older real column notebook may have crashed

If this notebook runs while the older one crashed, the likely cause was not simply row count. Plausible differences:

- the older notebook kept more real-data state alive while fitting;
- `df_map`, exact/grid maps, optimizer graph, and column grouped tensors may have overlapped in memory;
- a Jupyter kernel that already ran several heavy fits can have allocator fragmentation;
- the dense exact head block is still a risk factor, but high-resolution simulation showed `heads=2736` can be feasible when the state is clean.

This notebook loads only one coordinate version at a time for the fitting stage.